# Mission 2 Stage 4-A — 59D vs eGeMAPSv02 88D

질문: 현재 clean 59D AF10_AF11_COMBINATION과 openSMILE eGeMAPSv02 Functionals 88D를 동일한 M04-100 조건에서 비교하면 internal validation 성능과 오류 구조가 어떻게 달라지는가? clean 59D의 사용자 확인 기준은 Accuracy 0.873760이다.

한 번에 하나의 Checkpoint 셀만 직접 실행한다. B1/B5의 저장 output은 historical 감사 증거이며 C~J는 아직 실행되지 않았다. clean B4/B6 결과는 사용자 보고로 experiment log에 기록했다. 결과 해석과 다음 실험 결정은 사용자가 한다. 모델 입력은 현재 발화 waveform에서 추출한 feature뿐이다. record_key, call_id, speaker, crop 위치, duration 및 overlap 정보는 정렬·split·진단에만 사용한다.


## Checkpoint A — 환경과 고정 비교 계약

현재 브랜치와 소스 계약을 확인한다. 이 셀은 feature 추출이나 모델 학습을 시작하지 않는다.


In [22]:
from pathlib import Path
import hashlib
import subprocess
import sys
import warnings

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src" / "dcc_data").is_dir():
    raise RuntimeError("STOP: repository root could not be resolved")
sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import StandardScaler

from dcc_data.audio import read_wav_frames
from dcc_data.mission2_model_experiments import (
    EXPECTED_INTERNAL_SPLIT, EXPECTED_OFFICIAL_CALLS, EXPECTED_OFFICIAL_ROWS,
    M04_MLP_CONFIG, build_standalone_diagnostic_report,
    calculate_paired_prediction_changes, calculate_prediction_metrics,
    create_fixed_internal_split, load_standalone_e05_cache,
    record_key_signature, standalone_group_split_indices,
    standalone_training_cache_paths,
)
from dcc_data.mission2_acoustic_feature_expansion import (
    BLOCK_BY_ID, CANONICAL_52D_FEATURE_NAMES, CANONICAL_52D_PREPARED_ID,
    M04_ITER100_FEATURE_SCREENING_CONFIG, NEXT_CANDIDATE_BLOCK_IDS,
    ROUND1_BLOCK_IDS, ROUND2_BLOCK_IDS, RESIDUAL_TARGET_BLOCK_IDS,
    PreparedRepresentation, _prepare_representation, _run_prepared_representation,
    load_candidate_feature_cache, prepare_residual_combination_screening,
)
from dcc_data.mission2_standard_representation import (
    REPRESENTATION_ID, build_egemaps_cache, extract_egemaps,
    get_egemaps_feature_names, load_egemaps_cache,
)

RAW_DATA_ROOT = REPO_ROOT / "data" / "raw_original"
MANIFEST_PATH = REPO_ROOT / "tmp" / "mission2_eda" / "utterance_manifest_v1.parquet"
CANDIDATE_CACHE_ROOT = REPO_ROOT / "data" / "processed" / "mission2_acoustic_feature_expansion"
EGEMAPS_CACHE_PATH = (
    REPO_ROOT / "data" / "processed" / "mission2_standard_representation"
    / "training_egemaps_v02_88d.npz"
)
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_ROOT, text=True
).strip()
print("Repository:", REPO_ROOT)
print("Branch:", branch)
print("Clean 59D reference: AF10_AF11_COMBINATION, Accuracy 0.873760; classifier: M04-100")
print("Classifier config:", dict(M04_ITER100_FEATURE_SCREENING_CONFIG))
print("Expected internal split:", dict(EXPECTED_INTERNAL_SPLIT))


Repository: C:\Users\YW\Study\02_Projects\DCC_2026\prelim
Branch: yeonwoo/mission2-standard-representation
Clean 59D reference: AF10_AF11_COMBINATION, Accuracy 0.873760; classifier: M04-100
Classifier config: {'hidden_layer_sizes': (64,), 'activation': 'relu', 'solver': 'adam', 'alpha': 0.0001, 'batch_size': 1024, 'learning_rate': 'constant', 'learning_rate_init': 0.001, 'max_iter': 100, 'shuffle': True, 'random_state': 42, 'tol': 0.0001, 'early_stopping': False, 'n_iter_no_change': 10}
Expected internal split: {'total_rows': 889358, 'train_rows': 711467, 'validation_rows': 177891, 'train_calls': 23360, 'validation_calls': 5840, 'call_overlap': 0}


## Checkpoint B — 59D reference와 source provenance

B1은 historical cache의 상태를 보여준다. 모두 UNVERIFIABLE이면 승인하지 않는다. B2는 완료된 clean chain을 검증해 재사용하거나 없을 때만 새 versioned chain을 만든다. B3에서 clean chain을 strict-load하고 B4에서 clean 52D·59D를 재현한다. B5는 historical cache와 clean chain을 비교한다. B6는 E05 3개 차이를 확인하고 historical-cache 52D·59D를 한 번씩 재현한다. 기존 artifact와 historical score는 유지한다.


In [9]:
import json
from dcc_data.mission2_standard_representation import (
    PROVENANCE_EXACT, audit_historical_af09_divergence,
    audit_stage4_59d_dependencies,
    compare_manifest_inventories, find_historical_manifest_metadata,
    minimum_59d_rebuild_plan,
)
audit_59d = audit_stage4_59d_dependencies(REPO_ROOT)
display(audit_59d)
old_signature = "5cea8f5595d7ae02fea1dea3c4b8d26916902a91560ca44602e004cdfd3752c6"
old_manifests = find_historical_manifest_metadata(
    (REPO_ROOT / "tmp" / "mission2_eda", REPO_ROOT / "data" / "processed"),
    old_signature,
)
print("Historical manifest metadata candidates:", old_manifests)
for old_path in old_manifests:
    old_meta = json.loads(old_path.read_text(encoding="utf-8"))
    current_meta = json.loads(standalone_training_cache_paths(REPO_ROOT).manifest_metadata.read_text(encoding="utf-8"))
    inventory_audit = compare_manifest_inventories(old_meta, current_meta)
    print(old_path, inventory_audit["status"])
    display(pd.DataFrame(inventory_audit.get("differences", [])))
display(audit_59d.loc[audit_59d["status"] != PROVENANCE_EXACT,
                  ["cache", "status", "reason", "recommended_action"]])
print("Minimum dependent rebuild plan (do not overwrite historical artifacts):")
display(minimum_59d_rebuild_plan(audit_59d))


,cache,rows,keys,artifact_path,metadata_path,call_id,label,schema,schema_version,source_split,...,source_signature,manifest_source_signature,source_inventory_signature,source_content_sha256,record_key_signature,config_fingerprint,parent_provenance,status,recommended_action,reason
0,MFCC40,889358,exact,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,exact,exact,exact,mission2_mfcc_mean_std_v1,Training,...,NaN,ad768e26ab3b4b132d3505e1aa3b733b5387ff36dda535...,NaN,None,NaN,"{'hop_length': 160, 'n_fft': 512, 'n_mfcc': 20...",not recorded,UNVERIFIABLE,regenerate cache,RAW_SOURCE_IDENTITY_NOT_RECORDED
1,E05 candidate,889358,exact,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,exact,unavailable,exact,mission2_candidate_audio_features_v1,Training,...,5cea8f5595d7ae02fea1dea3c4b8d26916902a91560ca4...,ad768e26ab3b4b132d3505e1aa3b733b5387ff36dda535...,5cea8f5595d7ae02fea1dea3c4b8d26916902a91560ca4...,None,7c1e27f4f45ce84f261467f5863fcf8df838a8f1b22dc2...,"{'clipping_amplitude': 0.999, 'hop_length': 16...",not recorded,UNVERIFIABLE,regenerate cache,SOURCE_SIGNATURE_MISMATCH
2,AF02,889358,exact,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,exact,unavailable,exact,mission2_acoustic_feature_expansion_v1,Training,...,NaN,ad768e26ab3b4b132d3505e1aa3b733b5387ff36dda535...,NaN,None,7c1e27f4f45ce84f261467f5863fcf8df838a8f1b22dc2...,48b00f6cbc7657c35954b57f602e897157d2dbe2713542...,not recorded,UNVERIFIABLE,regenerate cache,RAW_SOURCE_IDENTITY_NOT_RECORDED
3,AF04,889358,exact,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,exact,unavailable,exact,mission2_acoustic_feature_expansion_v1,Training,...,NaN,ad768e26ab3b4b132d3505e1aa3b733b5387ff36dda535...,NaN,None,7c1e27f4f45ce84f261467f5863fcf8df838a8f1b22dc2...,4dfa73c9bf5149a5438c05b859db154d4296acd7a76bbb...,not recorded,UNVERIFIABLE,regenerate cache,RAW_SOURCE_IDENTITY_NOT_RECORDED
4,AF09,889358,exact,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,exact,unavailable,exact,mission2_acoustic_feature_expansion_v1,Training,...,NaN,ad768e26ab3b4b132d3505e1aa3b733b5387ff36dda535...,NaN,None,7c1e27f4f45ce84f261467f5863fcf8df838a8f1b22dc2...,43857dc95c1cf6cacc184576caa79d5d64de86ea66ef8d...,not recorded,UNVERIFIABLE,regenerate cache,RAW_SOURCE_IDENTITY_NOT_RECORDED
5,AF10/AF11,889358,exact,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,exact,unavailable,exact,mission2_acoustic_feature_expansion_v1,Training,...,NaN,ad768e26ab3b4b132d3505e1aa3b733b5387ff36dda535...,NaN,None,7c1e27f4f45ce84f261467f5863fcf8df838a8f1b22dc2...,1c15246dd635bba231a354c979ac2c0e186c16ce99d5a1...,not recorded,UNVERIFIABLE,regenerate cache,RAW_SOURCE_IDENTITY_NOT_RECORDED


Historical manifest metadata candidates: []


,cache,status,reason,recommended_action
0,MFCC40,UNVERIFIABLE,RAW_SOURCE_IDENTITY_NOT_RECORDED,regenerate cache
1,E05 candidate,UNVERIFIABLE,SOURCE_SIGNATURE_MISMATCH,regenerate cache
2,AF02,UNVERIFIABLE,RAW_SOURCE_IDENTITY_NOT_RECORDED,regenerate cache
3,AF04,UNVERIFIABLE,RAW_SOURCE_IDENTITY_NOT_RECORDED,regenerate cache
4,AF09,UNVERIFIABLE,RAW_SOURCE_IDENTITY_NOT_RECORDED,regenerate cache
5,AF10/AF11,UNVERIFIABLE,RAW_SOURCE_IDENTITY_NOT_RECORDED,regenerate cache


Minimum dependent rebuild plan (do not overwrite historical artifacts):


,cache,status,dependent_rebuild,action,preserve_existing_artifact
0,MFCC40,UNVERIFIABLE,MFCC40 → E05 42D → canonical 52D → 59D,run Stage 4 B2 clean versioned chain builder,True
1,E05 candidate,UNVERIFIABLE,E05 candidate → E05 42D → canonical 52D → 59D,run Stage 4 B2 with current v2 candidate seman...,True
2,AF02,UNVERIFIABLE,AF02 → canonical 52D → 59D,run Stage 4 B2 with the unchanged AF02 block,True
3,AF04,UNVERIFIABLE,AF04 → canonical 52D → 59D,run Stage 4 B2 with the unchanged AF04 block,True
4,AF09,UNVERIFIABLE,AF09 → canonical 52D → 59D,run Stage 4 B2 with the unchanged AF09 block,True
5,AF10/AF11,UNVERIFIABLE,AF10/AF11 → 59D,run Stage 4 B2 with the unchanged AF10/AF11 bl...,True


### Checkpoint B2 — clean cache build (직접 실행)

이 셀은 기존 완료 clean chain이 하나 있으면 strict validation 후 재사용한다. 없을 때만 현재 raw Training에서 MFCC40, v2 E05 candidate, AF02, AF04, AF09, AF10, AF11을 새 versioned directory에 추출한다. 기존 체인이 불완전하거나 현재 source와 맞지 않으면 중단한다. 기존 cache와 score는 수정하지 않는다.

mtime inventory와 stable row signature를 분리해 기록한다. 전체 WAV/JSON 내용 hash는 별도 비용이 큰 감사이므로 이 셀은 내용 동일성을 과거 cache에 소급 주장하지 않는다.


In [10]:
from dcc_data.mission2_clean_stage4 import load_or_build_clean_stage4_chain
CLEAN_CACHE_ROOT = REPO_ROOT / "data" / "processed" / "mission2_stage4_clean"
clean_59d, clean_chain_status = load_or_build_clean_stage4_chain(
    MANIFEST_PATH, standalone_training_cache_paths(REPO_ROOT).manifest_metadata,
    RAW_DATA_ROOT, CLEAN_CACHE_ROOT,
)
CLEAN_CHAIN_ROOT = clean_59d.chain_root
print(clean_chain_status, CLEAN_CHAIN_ROOT)
print("Provenance fingerprint:", clean_59d.provenance_fingerprint)


REUSING_EXISTING_CLEAN_CHAIN C:\Users\YW\Study\02_Projects\DCC_2026\prelim\data\processed\mission2_stage4_clean\stage4_clean_v1_090422f783f0d911
Provenance fingerprint: 49d65a35d1e98c280f98e45cbed864ef8bac36268ccffbe81aa0985923c90de3


### Checkpoint B3 — clean 59D strict validation

B2가 완료된 후 새 체인만 읽는다. 모든 artifact의 metadata·부모·파일 hash와 현재 manifest/inventory를 재확인하고, fit 전에 59D 행·순서·split을 고정한다.


In [11]:
from dcc_data.mission2_clean_stage4 import load_clean_stage4_59d
if "CLEAN_CHAIN_ROOT" not in globals():
    raise RuntimeError("STOP: run Checkpoint B2 clean build first")
clean_59d = load_clean_stage4_59d(
    CLEAN_CHAIN_ROOT, MANIFEST_PATH,
    standalone_training_cache_paths(REPO_ROOT).manifest_metadata,
    RAW_DATA_ROOT,
)
keys, calls, labels = clean_59d.record_keys, clean_59d.call_ids, clean_59d.y
manifest = pd.read_parquet(MANIFEST_PATH)
training_manifest = manifest.loc[manifest["split"].astype(str) == "Training"].copy()
aligned_manifest = training_manifest.set_index("record_key", verify_integrity=True).loc[keys].reset_index()
tracking = aligned_manifest.loc[:, [
    "record_key", "call_id", "annotation_row", "speaker", "startAt_ms",
    "endAt_ms", "duration_sec",
]].copy()
tracking["duration_ms"] = tracking["endAt_ms"] - tracking["startAt_ms"]
train_idx, validation_idx, split_audit = create_fixed_internal_split(
    calls, split_function=standalone_group_split_indices,
    expected_contract=EXPECTED_INTERNAL_SPLIT,
)
if split_audit["call_overlap"] != 0 or split_audit["seed"] != 42:
    raise RuntimeError("STOP: fixed GroupSplit contract differs")
prepared_59d = _prepare_representation(
    representation_id="AF10_AF11_COMBINATION",
    X=clean_59d.X, y=labels, record_keys=keys,
    feature_names=clean_59d.feature_names,
    train_idx=train_idx, validation_idx=validation_idx,
    split_audit=split_audit,
)
source_provenance = clean_59d.provenance_fingerprint
EGEMAPS_CACHE_PATH = CLEAN_CHAIN_ROOT / "training_egemaps_v02_88d.npz"
print("Clean 59D:", clean_59d.X.shape, clean_59d.X.dtype)
print("Provenance fingerprint:", source_provenance)
display(pd.DataFrame([split_audit]))

C:\Users\YW\AppData\Local\Temp\ipykernel_30996\4119950183.py:12: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  aligned_manifest = training_manifest.set_index("record_key", verify_integrity=True).loc[keys].reset_index()


Clean 59D: (889358, 59) float32
Provenance fingerprint: 49d65a35d1e98c280f98e45cbed864ef8bac36268ccffbe81aa0985923c90de3


,seed,test_size,total_rows,train_rows,validation_rows,train_calls,validation_calls,call_overlap
0,42,0.2,889358,711467,177891,23360,5840,0


### Checkpoint B4 — clean AF09 52D → 59D M04-100 reproduction

같은 fixed call_id split, seed 42, internal-train-only scaler, M04-100 설정으로 AF09 canonical 52D를 먼저 fit한다. 최초 historical 52D와 canonical historical 52D의 Accuracy·confusion matrix·Recall을 각각 비교한 뒤 59D를 fit하고 historical 59D와 비교한다. 일치 여부는 진단 결과이며 자동 canonical 승격이나 divergence 원인 단정은 하지 않는다.


In [12]:
from dcc_data.mission2_acoustic_feature_expansion import (
    CANONICAL_52D_CONFUSION_MATRIX, CANONICAL_52D_REFERENCE_ID,
    HISTORICAL_NON_REPRODUCED_52D_CONFUSION_MATRIX,
    validate_canonical_52d_prepared,
)
expected_m04 = {**dict(M04_MLP_CONFIG), "max_iter": 100}
if dict(M04_ITER100_FEATURE_SCREENING_CONFIG) != expected_m04:
    raise RuntimeError("STOP: M04-100 configuration differs")
if (clean_59d.X.shape != (EXPECTED_OFFICIAL_ROWS["Training"], 59)
        or clean_59d.X.dtype != np.float32
        or tuple(clean_59d.feature_names[:52]) != CANONICAL_52D_FEATURE_NAMES
        or not np.isfinite(clean_59d.X).all()):
    raise RuntimeError("STOP: clean AF09 52D prefix contract differs")
prepared_52d = _prepare_representation(
    representation_id=CANONICAL_52D_PREPARED_ID,
    X=clean_59d.X[:, :52], y=labels, record_keys=keys,
    feature_names=CANONICAL_52D_FEATURE_NAMES,
    train_idx=train_idx, validation_idx=validation_idx, split_audit=split_audit,
)
validate_canonical_52d_prepared(
    prepared_52d, canonical_matrix=clean_59d.X[:, :52],
    canonical_labels=labels, canonical_record_keys=keys,
    canonical_train_idx=train_idx, canonical_validation_idx=validation_idx,
    expected_split_contract=EXPECTED_INTERNAL_SPLIT,
)
if split_audit["call_overlap"] != 0 or split_audit["seed"] != 42:
    raise RuntimeError("STOP: fixed call_id split differs before fit")

def check_clean_m04_result(result, prepared):
    if (dict(result.model_config) != expected_m04
            or result.fit_audit["scaler_fit_scope"] != "internal_train_only"
            or result.fit_audit["seed"] != 42
            or not np.array_equal(result.fit_audit["train_idx"], train_idx)
            or not np.array_equal(result.fit_audit["validation_idx"], validation_idx)
            or not np.array_equal(result.y_true, labels[validation_idx])
            or tuple(result.feature_names) != tuple(prepared.feature_names)):
        raise RuntimeError("STOP: clean M04-100 fit contract differs")

def compare_historical_metrics(result, reference, historical_accuracy, historical_cm):
    historical_cm = np.asarray(historical_cm, dtype=np.int64)
    observed_cm = np.asarray(result.metrics["confusion_matrix"], dtype=np.int64)
    if historical_cm.shape != (2, 2) or historical_cm.sum() != len(validation_idx):
        raise RuntimeError(f"STOP: historical confusion matrix contract differs: {reference}")
    historical_recalls = historical_cm.diagonal() / historical_cm.sum(axis=1)
    accuracy_delta = result.metrics["accuracy"] - historical_accuracy
    cm_exact = np.array_equal(observed_cm, historical_cm)
    # Canonical 52D의 저장된 Accuracy는 CM 계산값과 1 ULP 다르므로 점수만 미세 허용한다.
    return {
        "reference": reference,
        "historical_accuracy": historical_accuracy,
        "clean_accuracy": result.metrics["accuracy"],
        "accuracy_delta": accuracy_delta,
        "historical_confusion_matrix": historical_cm.tolist(),
        "clean_confusion_matrix": observed_cm.tolist(),
        "historical_cm_exact": cm_exact,
        "historical_recall_0": float(historical_recalls[0]),
        "historical_recall_1": float(historical_recalls[1]),
        "clean_recall_0": result.metrics["recall_0"],
        "clean_recall_1": result.metrics["recall_1"],
        "recall_delta_0": result.metrics["recall_0"] - historical_recalls[0],
        "recall_delta_1": result.metrics["recall_1"] - historical_recalls[1],
        "status": ("REPRODUCED_FROM_CLEAN_PROVENANCE"
                   if cm_exact and abs(accuracy_delta) <= 1e-12
                   else "HISTORICAL_NON_REPRODUCED"),
        "provenance_fingerprint": source_provenance,
    }

result_52d = _run_prepared_representation(
    prepared_52d, experiment_id=CANONICAL_52D_REFERENCE_ID, allow_fit=True
)
check_clean_m04_result(result_52d, prepared_52d)
train_metrics_52d = calculate_prediction_metrics(
    prepared_52d.y_train, result_52d.model.predict(prepared_52d.X_train_scaled),
)
report_52d = {
    "representation": "AF09_52D_REFERENCE", "dimension": 52,
    "train_accuracy": train_metrics_52d["accuracy"],
    "internal_validation_accuracy": result_52d.metrics["accuracy"],
    "generalization_gap": train_metrics_52d["accuracy"] - result_52d.metrics["accuracy"],
    "confusion_matrix": result_52d.metrics["confusion_matrix"],
    "recall_0": result_52d.metrics["recall_0"],
    "recall_1": result_52d.metrics["recall_1"],
    "correct_count": result_52d.metrics["correct_count"],
    "error_count": len(validation_idx) - result_52d.metrics["correct_count"],
    "provenance_fingerprint": source_provenance,
}
af09_comparisons = pd.DataFrame([
    compare_historical_metrics(
        result_52d, "AF09_first_historical", 0.8689703245245685,
        HISTORICAL_NON_REPRODUCED_52D_CONFUSION_MATRIX,
    ),
    compare_historical_metrics(
        result_52d, "AF09_canonical_historical", 0.8695324665103912,
        CANONICAL_52D_CONFUSION_MATRIX,
    ),
])
print("Clean AF09 52D M04-100:")
display(pd.DataFrame([report_52d]))
display(af09_comparisons)

if (prepared_59d.feature_names != clean_59d.feature_names
        or not np.array_equal(prepared_59d.train_idx, train_idx)
        or not np.array_equal(prepared_59d.validation_idx, validation_idx)
        or int(prepared_59d.scaler.n_samples_seen_) != len(train_idx)
        or split_audit["call_overlap"] != 0 or split_audit["seed"] != 42):
    raise RuntimeError("STOP: clean 59D split/scaler contract differs before fit")
result_59d = _run_prepared_representation(
    prepared_59d, experiment_id="AF10_AF11_COMBINATION", allow_fit=True
)
check_clean_m04_result(result_59d, prepared_59d)
train_metrics_59d = calculate_prediction_metrics(
    prepared_59d.y_train, result_59d.model.predict(prepared_59d.X_train_scaled),
)
comparison_59d = compare_historical_metrics(
    result_59d, "AF10_AF11_59D_historical", 0.8745580158636468,
    ((71421, 13930), (8385, 84155)),
)
report_59d = {
    "representation": "AF10_AF11_COMBINATION", "dimension": 59,
    "train_accuracy": train_metrics_59d["accuracy"],
    "internal_validation_accuracy": result_59d.metrics["accuracy"],
    "generalization_gap": train_metrics_59d["accuracy"] - result_59d.metrics["accuracy"],
    "status": comparison_59d["status"],
    "accuracy": result_59d.metrics["accuracy"],
    "historical_accuracy": comparison_59d["historical_accuracy"],
    "accuracy_delta": comparison_59d["accuracy_delta"],
    "confusion_matrix": result_59d.metrics["confusion_matrix"],
    "historical_cm_exact": comparison_59d["historical_cm_exact"],
    "recall_0": result_59d.metrics["recall_0"],
    "recall_1": result_59d.metrics["recall_1"],
    "correct_count": result_59d.metrics["correct_count"],
    "error_count": len(validation_idx) - result_59d.metrics["correct_count"],
    "provenance_fingerprint": source_provenance,
}
print("Clean AF10+AF11 59D M04-100:")
display(pd.DataFrame([report_59d]))
display(pd.DataFrame([comparison_59d]))
print("AF09 historical divergence:", audit_historical_af09_divergence({}, {})["status"])


Clean AF09 52D M04-100:


,representation,dimension,train_accuracy,internal_validation_accuracy,generalization_gap,confusion_matrix,recall_0,recall_1,correct_count,error_count,provenance_fingerprint
0,AF09_52D_REFERENCE,52,0.87049,0.869594,0.000896,"[[71215, 14136], [9062, 83478]]",0.834378,0.902075,154693,23198,49d65a35d1e98c280f98e45cbed864ef8bac36268ccffb...


,reference,historical_accuracy,clean_accuracy,accuracy_delta,historical_confusion_matrix,clean_confusion_matrix,historical_cm_exact,historical_recall_0,historical_recall_1,clean_recall_0,clean_recall_1,recall_delta_0,recall_delta_1,status,provenance_fingerprint
0,AF09_first_historical,0.868970,0.869594,0.000624,"[[70926, 14425], [8884, 83656]]","[[71215, 14136], [9062, 83478]]",False,0.830992,0.903998,0.834378,0.902075,0.003386,-0.001923,HISTORICAL_NON_REPRODUCED,49d65a35d1e98c280f98e45cbed864ef8bac36268ccffb...
1,AF09_canonical_historical,0.869532,0.869594,0.000062,"[[71066, 14285], [8924, 83616]]","[[71215, 14136], [9062, 83478]]",False,0.832632,0.903566,0.834378,0.902075,0.001746,-0.001491,HISTORICAL_NON_REPRODUCED,49d65a35d1e98c280f98e45cbed864ef8bac36268ccffb...


Clean AF10+AF11 59D M04-100:


,representation,dimension,train_accuracy,internal_validation_accuracy,generalization_gap,status,accuracy,historical_accuracy,accuracy_delta,confusion_matrix,historical_cm_exact,recall_0,recall_1,correct_count,error_count,provenance_fingerprint
0,AF10_AF11_COMBINATION,59,0.875788,0.87376,0.002028,HISTORICAL_NON_REPRODUCED,0.87376,0.874558,-0.000798,"[[72265, 13086], [9371, 83169]]",False,0.84668,0.898736,155434,22457,49d65a35d1e98c280f98e45cbed864ef8bac36268ccffb...


,reference,historical_accuracy,clean_accuracy,accuracy_delta,historical_confusion_matrix,clean_confusion_matrix,historical_cm_exact,historical_recall_0,historical_recall_1,clean_recall_0,clean_recall_1,recall_delta_0,recall_delta_1,status,provenance_fingerprint
0,AF10_AF11_59D_historical,0.874558,0.87376,-0.000798,"[[71421, 13930], [8385, 84155]]","[[72265, 13086], [9371, 83169]]",False,0.836792,0.909391,0.84668,0.898736,0.009889,-0.010655,HISTORICAL_NON_REPRODUCED,49d65a35d1e98c280f98e45cbed864ef8bac36268ccffb...


AF09 historical divergence: UNRESOLVED_DUE_TO_MISSING_ARTIFACT


### Checkpoint B5 — Historical vs Clean Feature Audit

이 셀 하나만 직접 실행한다. 정확한 clean provenance fingerprint를 가진 기존 체인과 고정 historical cache 경로만 읽는다. MFCC40, E05, AF02, AF04, AF09, AF10/AF11의 float32 모델 입력 차이를 비교하고 52D·59D 연결을 요약한다. historical cache의 원래 dtype·저장 열 순서·raw 값 exact 여부도 별도로 표시한다. 허용 오차 비교는 exact 판정을 바꾸지 않는다. cache 생성, 추출, 모델 fit, 원인 확정은 하지 않는다.


In [13]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display
b5_repo_root = Path.cwd().resolve()
if b5_repo_root.name == "notebooks":
    b5_repo_root = b5_repo_root.parent
if not (b5_repo_root / "src" / "dcc_data").is_dir():
    raise RuntimeError("STOP: repository root could not be resolved")
sys.path.insert(0, str(b5_repo_root / "src"))
from dcc_data.mission2_standard_representation import audit_historical_vs_clean_feature_caches
b5_expected_fingerprint = "49d65a35d1e98c280f98e45cbed864ef8bac36268ccffbe81aa0985923c90de3"
b5_cache_root = b5_repo_root / "data" / "processed" / "mission2_stage4_clean"
b5_matches = []
for b5_path in b5_cache_root.glob("stage4_clean_v1_*"):
    b5_chain_file = b5_path / "chain.json"
    if not b5_chain_file.is_file():
        continue
    try:
        b5_chain = json.loads(b5_chain_file.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        continue
    if b5_chain.get("provenance_fingerprint") == b5_expected_fingerprint:
        b5_matches.append(b5_path)
if len(b5_matches) != 1:
    raise RuntimeError("STOP: exactly one clean chain must match the B4 fingerprint")
b5_audit = audit_historical_vs_clean_feature_caches(
    b5_repo_root, b5_matches[0], expected_fingerprint=b5_expected_fingerprint,
)
display(b5_audit["block_audit"])
display(pd.DataFrame([b5_audit["assembled_52d"]]))
display(pd.DataFrame([b5_audit["assembled_59d"]]))
print(b5_audit["conclusion"])


,block,historical_artifact_path,clean_artifact_path,historical_row_count,clean_row_count,historical_feature_dimension,clean_feature_dimension,historical_stored_feature_dimension,clean_stored_feature_dimension,record_key_exact,...,raw_cache_exact_equality,exact_feature_matrix_equality,numerical_close,differing_row_count,differing_value_count,mean_abs_diff,max_abs_diff,top_changed_features,status,reason
0,MFCC40,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,889358,889358,40,40,40,40,True,...,True,True,True,0,0,0.000000e+00,0.000000,[],COMPARABLE,None
1,E05 candidate,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,889358,889358,2,2,5,2,True,...,False,False,False,3,3,2.312692e-07,0.156807,"[{'feature': 'zcr', 'differing_row_count': 3, ...",COMPARABLE,None
2,AF02,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,889358,889358,5,5,11,5,True,...,True,True,True,0,0,0.000000e+00,0.000000,[],COMPARABLE,None
3,AF04,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,889358,889358,2,2,4,2,True,...,True,True,True,0,0,0.000000e+00,0.000000,[],COMPARABLE,None
4,AF09,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,889358,889358,3,3,7,3,True,...,True,True,True,0,0,0.000000e+00,0.000000,[],COMPARABLE,None
5,AF10/AF11,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...,889358,889358,7,7,10,7,True,...,True,True,True,0,0,0.000000e+00,0.000000,[],COMPARABLE,None


,historical_shape,clean_shape,feature_order_equal,historical_dtype,clean_dtype,comparison_dtype,status,exact_feature_matrix_equality,numerical_close,differing_row_count,differing_value_count,mean_abs_diff,max_abs_diff,changed_blocks
0,"(889358, 52)","(889358, 52)",True,float32,float32,float32,COMPARABLE,False,False,3,3,8.894970e-09,0.156807,[E05 candidate]


,historical_shape,clean_shape,feature_order_equal,historical_dtype,clean_dtype,comparison_dtype,status,exact_feature_matrix_equality,numerical_close,differing_row_count,differing_value_count,mean_abs_diff,max_abs_diff,changed_blocks
0,"(889358, 59)","(889358, 59)",True,float32,float32,float32,COMPARABLE,False,False,3,3,7.839635e-09,0.156807,[E05 candidate]


{'status': 'FEATURE_CACHE_DIFFERENCE_FOUND', 'changed_model_input_blocks': ['E05 candidate'], 'raw_only_changed_blocks': [], 'interpretation': 'Selected float32 model inputs differ; feature caches are a cause candidate, but historical prediction/model artifacts are needed for a causal conclusion.', 'clean_provenance_fingerprint': '49d65a35d1e98c280f98e45cbed864ef8bac36268ccffbe81aa0985923c90de3', 'historical_raw_content_identity': 'UNVERIFIABLE', 'rtol': 1e-07, 'atol': 1e-08}


### Checkpoint B6 — E05 Historical Divergence Check (직접 실행)

현재 clean chain과 historical cache를 읽기 전용으로 다시 검증한다. E05에서 달라진 3개 값의 행·통화·label·fixed split 소속을 먼저 출력한다. 안전하게 조립되면 historical-cache 52D와 59D를 동일 call_id split, seed 42, internal-train-only scaler, M04-100으로 각각 한 번만 fit한다. 과거 Accuracy·CM·Recall과 비교한 뒤 forensic을 종료한다. 당시 prediction/checkpoint가 없다면 인과를 완전히 증명한 것으로 해석하지 않는다.


In [14]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
b6_root = Path.cwd().resolve()
if b6_root.name == "notebooks":
    b6_root = b6_root.parent
if not (b6_root / "src" / "dcc_data").is_dir():
    raise RuntimeError("STOP: repository root could not be resolved")
sys.path.insert(0, str(b6_root / "src"))
from dcc_data.mission2_standard_representation import (
    audit_historical_vs_clean_feature_caches, annotate_e05_fixed_split,
    reproduce_historical_stage4_inputs,
)
from dcc_data.mission2_model_experiments import (
    create_fixed_internal_split, standalone_group_split_indices, EXPECTED_INTERNAL_SPLIT,
)
from dcc_data.mission2_acoustic_feature_expansion import (
    CANONICAL_52D_CONFUSION_MATRIX,
    HISTORICAL_NON_REPRODUCED_52D_CONFUSION_MATRIX,
)
b6_chain = b6_root / "data/processed/mission2_stage4_clean/stage4_clean_v1_090422f783f0d911"
b6_fingerprint = "49d65a35d1e98c280f98e45cbed864ef8bac36268ccffbe81aa0985923c90de3"
b6_audit = audit_historical_vs_clean_feature_caches(
    b6_root, b6_chain, expected_fingerprint=b6_fingerprint,
    include_reproduction_inputs=True,
)
b6_inputs = b6_audit["reproduction_inputs"]
if b6_inputs is None:
    print("INCOMPLETE_HISTORICAL_EVIDENCE")
    display(b6_audit["block_audit"].loc[lambda frame: frame["status"] != "COMPARABLE"])
else:
    b6_changed = b6_audit["conclusion"]["changed_model_input_blocks"]
    b6_e05 = b6_inputs["e05_differences"]
    if (b6_changed != ["E05 candidate"] or len(b6_e05) != 3
            or b6_audit["assembled_52d"]["differing_row_count"] != 3
            or b6_audit["assembled_59d"]["differing_row_count"] != 3
            or b6_audit["assembled_52d"]["differing_value_count"] != 3
            or b6_audit["assembled_59d"]["differing_value_count"] != 3):
        raise RuntimeError("STOP: B5 three-value E05 contract changed; no fit")
    b6_train, b6_validation, b6_split = create_fixed_internal_split(
        b6_inputs["call_ids"], split_function=standalone_group_split_indices,
        expected_contract=EXPECTED_INTERNAL_SPLIT,
    )
    b6_rows = annotate_e05_fixed_split(
        b6_e05, b6_train, b6_validation, total_rows=len(b6_inputs["y"]),
    )
    display(b6_rows.loc[:, ["record_key", "call_id", "label",
                            "fixed_split_membership", "feature_name",
                            "historical_value", "clean_value",
                            "absolute_difference"]])
    b6_reproduction = reproduce_historical_stage4_inputs(b6_inputs)
    b6_52, b6_59 = b6_reproduction["results"]
    b6_references = (
        ("AF09 first 52D", b6_52, 0.8689703245245685,
         HISTORICAL_NON_REPRODUCED_52D_CONFUSION_MATRIX),
        ("AF09 canonical 52D", b6_52, 0.8695324665103912,
         CANONICAL_52D_CONFUSION_MATRIX),
        ("AF10_AF11 59D", b6_59, 0.8745580158636468,
         ((71421, 13930), (8385, 84155))),
    )
    b6_reports = []
    for b6_name, b6_result, b6_historical_accuracy, b6_historical_cm in b6_references:
        b6_metrics = b6_result.metrics
        b6_cm = np.asarray(b6_metrics["confusion_matrix"], dtype=np.int64)
        b6_historical_cm = np.asarray(b6_historical_cm, dtype=np.int64)
        if b6_historical_cm.sum() != len(b6_validation):
            raise RuntimeError("STOP: historical CM evaluation size differs")
        b6_cm_exact = bool(np.array_equal(b6_cm, b6_historical_cm))
        b6_accuracy_exact = bool(b6_metrics["accuracy"] == b6_historical_accuracy)
        b6_accuracy_close = abs(b6_metrics["accuracy"] - b6_historical_accuracy) <= 1e-12
        b6_match = b6_cm_exact and b6_accuracy_exact
        b6_reports.append({
            "reference": b6_name, "accuracy": b6_metrics["accuracy"],
            "historical_accuracy": b6_historical_accuracy,
            "accuracy_delta": b6_metrics["accuracy"] - b6_historical_accuracy,
            "accuracy_literal_exact": b6_accuracy_exact,
            "accuracy_within_1e-12": b6_accuracy_close,
            "confusion_matrix": b6_cm.tolist(),
            "historical_confusion_matrix": b6_historical_cm.tolist(),
            "cm_exact": b6_cm_exact, "historical_match": b6_match,
            "recall_0": b6_metrics["recall_0"],
            "recall_1": b6_metrics["recall_1"],
            "historical_recall_0": b6_historical_cm[0, 0] / b6_historical_cm[0].sum(),
            "historical_recall_1": b6_historical_cm[1, 1] / b6_historical_cm[1].sum(),
            "correct_count": b6_metrics["correct_count"],
            "error_count": len(b6_validation) - b6_metrics["correct_count"],
        })
    display(pd.DataFrame(b6_reports))
    b6_status = ("HISTORICAL_DIVERGENCE_EXPLAINED_BY_STORED_FEATURE_DIFFERENCE"
                 if any(row["historical_match"] for row in b6_reports)
                 else "FEATURE_DIFFERENCE_FOUND_BUT_HISTORICAL_SCORE_NOT_REPRODUCED")
    print(b6_status)
    print("Clean provenance fingerprint:", b6_reproduction["clean_provenance_fingerprint"])
    print("Historical raw content identity: UNVERIFIABLE; prediction/checkpoint evidence unavailable")


,record_key,call_id,label,fixed_split_membership,feature_name,historical_value,clean_value,absolute_difference
0,Training|651e4e712e98ee7120968df5_20220704|28,651e4e712e98ee7120968df5_20220704,1,internal-train,zcr,0.012207,0.169014,0.156807
1,Training|651e4f531e44dfea6eb05a69_20220901|3,651e4f531e44dfea6eb05a69_20220901,0,internal-train,zcr,0.010254,0.113924,0.103670
2,Training|651e5500835c8a556d7e2a8d_20220404|40,651e5500835c8a556d7e2a8d_20220404,0,internal-train,zcr,0.013672,0.164557,0.150885


,reference,accuracy,historical_accuracy,accuracy_delta,accuracy_literal_exact,accuracy_within_1e-12,confusion_matrix,historical_confusion_matrix,cm_exact,historical_match,recall_0,recall_1,historical_recall_0,historical_recall_1,correct_count,error_count
0,AF09 first 52D,0.869532,0.868970,0.000562,False,False,"[[71066, 14285], [8924, 83616]]","[[70926, 14425], [8884, 83656]]",False,False,0.832632,0.903566,0.830992,0.903998,154682,23209
1,AF09 canonical 52D,0.869532,0.869532,0.000000,True,True,"[[71066, 14285], [8924, 83616]]","[[71066, 14285], [8924, 83616]]",True,True,0.832632,0.903566,0.832632,0.903566,154682,23209
2,AF10_AF11 59D,0.874558,0.874558,0.000000,True,True,"[[71421, 13930], [8385, 84155]]","[[71421, 13930], [8385, 84155]]",True,True,0.836792,0.909391,0.836792,0.909391,155576,22315


HISTORICAL_DIVERGENCE_EXPLAINED_BY_STORED_FEATURE_DIFFERENCE
Clean provenance fingerprint: 49d65a35d1e98c280f98e45cbed864ef8bac36268ccffbe81aa0985923c90de3
Historical raw content identity: UNVERIFIABLE; prediction/checkpoint evidence unavailable


### Historical provenance 조사 종료 — 사용자 확인 결과

B5에서 old/clean model input 차이는 E05 zcr 3개 값뿐이었고, 모두 internal-train 행이었다. B6에서 old cache를 같은 split·scaler·M04-100으로 fit하자 당시 canonical 52D `0.8695324665103912`와 59D `0.8745580158636468`의 Accuracy·CM이 exact 재현됐다. 최초 AF09 52D `0.8689703245245685`는 미재현·원인 미해결이다. 당시 raw content와 prediction/checkpoint를 완전히 증명할 artifact는 없다. 이후 Stage 4는 clean 59D `0.873760`만 eGeMAPS의 비교 기준으로 사용한다. 이 문장은 사용자 보고 결과이며 이 Notebook에 B6 실행 output을 새로 만들어 넣은 것은 아니다.


## Checkpoint C — clean 59D 기준 확인과 소수 발화·synthetic 입력 호환성

B3/B4의 clean 59D provenance와 사용자 확인 기준을 먼저 확인한 뒤 실제 발화 소수만 읽기 전용으로 확인한다. 짧은 synthetic 신호는 NaN 대체 없이 오류가 나야 하며, 무음의 88D 값은 유한해야 한다. 8 kHz는 그대로 사용한다. 8 kHz Nyquist가 4 kHz이므로 openSMILE의 더 높은 대역 정의는 전체 실험 전 해석상 검토가 필요하다.


In [15]:
CLEAN_REFERENCE_FINGERPRINT = "49d65a35d1e98c280f98e45cbed864ef8bac36268ccffbe81aa0985923c90de3"
CLEAN_REFERENCE_CM = ((72265, 13086), (9371, 83169))
CLEAN_REFERENCE_ACCURACY = 155434 / 177891  # user-reported CM의 exact ratio
if "result_59d" not in globals():
    raise RuntimeError("STOP: run B4 clean 59D reproduction before eGeMAPS")
if (clean_59d.chain_root.name != "stage4_clean_v1_090422f783f0d911"
        or clean_59d.provenance_fingerprint != CLEAN_REFERENCE_FINGERPRINT
        or source_provenance != CLEAN_REFERENCE_FINGERPRINT
        or not np.array_equal(np.asarray(result_59d.metrics["confusion_matrix"]),
                              np.asarray(CLEAN_REFERENCE_CM))
        or abs(result_59d.metrics["accuracy"] - CLEAN_REFERENCE_ACCURACY) > 1e-12
        or tuple(result_59d.feature_names) != clean_59d.feature_names
        or result_59d.fit_audit["seed"] != 42
        or result_59d.fit_audit["scaler_fit_scope"] != "internal_train_only"
        or not np.array_equal(result_59d.fit_audit["train_idx"], train_idx)
        or not np.array_equal(result_59d.fit_audit["validation_idx"], validation_idx)):
    raise RuntimeError("STOP: current clean 59D reference chain differs")
feature_names_88 = get_egemaps_feature_names()
if len(feature_names_88) != 88 or len(set(feature_names_88)) != 88:
    raise ValueError("STOP: eGeMAPSv02 feature name contract differs")
sample_rows = aligned_manifest.loc[(aligned_manifest["crop_end_frame"] - aligned_manifest["crop_start_frame"]) >= 8_000].head(3)
if len(sample_rows) != 3:
    raise ValueError("STOP: compatibility sample is incomplete")
for row in sample_rows.itertuples():
    audio = read_wav_frames(RAW_DATA_ROOT / str(row.audio_path))
    if audio.sample_rate != 8_000 or audio.waveform.shape[0] != 1:
        raise ValueError("STOP: expected native 8 kHz mono WAV")
    crop = audio.waveform[
        0, int(row.crop_start_frame):int(row.crop_end_frame)
    ]
    values, names = extract_egemaps(crop, audio.sample_rate)
    if names != feature_names_88 or values.shape != (88,) or not np.isfinite(values).all():
        raise ValueError("STOP: small real utterance eGeMAPS contract failed")
    print("record_key / samples / dim:", row.record_key, len(crop), len(values))
silence, _ = extract_egemaps(np.zeros(8_000, dtype=np.float32), 8_000)
if not np.isfinite(silence).all():
    raise ValueError("STOP: silence produced NaN or inf")
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Segment too short")
    try:
        extract_egemaps(np.zeros(80, dtype=np.float32), 8_000)
    except ValueError as error:
        print("Expected very-short failure:", error)
    else:
        raise ValueError("STOP: very-short input was silently accepted")


record_key / samples / dim: Training|651e464d69a4f266f062680b_20220101|0 11336 88
record_key / samples / dim: Training|651e464d69a4f266f062680b_20220101|1 36240 88
record_key / samples / dim: Training|651e464d69a4f266f062680b_20220101|2 20552 88
Expected very-short failure: eGeMAPSv02 returned NaN or inf; audio may be too short


## Checkpoint C2 — Real Short-Audio eGeMAPS Preflight

사용자가 실행한 Training 889,358행의 preflight에서 가장 짧은 실제 발화 100개 중 72·80·80 samples의 3개가 실패했고, 성공한 가장 짧은 길이는 560 samples였다. 실제 실패 3개에 대한 후보 160/320/400은 실패, 800/1600/3200은 finite 88D에 성공했다. 800은 **시험한 후보 중 이 3개 모두에서 성공한 가장 작은 길이**이며 이론적 최소 길이 또는 전체 데이터 보장은 아니다.


In [20]:
# Manifest만으로 전체 Training의 crop 길이를 계산한다. WAV는 아래의 짧은 100개에 대해서만 읽는다.
if len(aligned_manifest) != len(keys) or len(aligned_manifest) != len(labels) or len(aligned_manifest) != len(calls):
    raise ValueError("STOP: preflight row counts differ from the aligned reference")
if not np.array_equal(aligned_manifest["record_key"].to_numpy(), np.asarray(keys)):
    raise ValueError("STOP: preflight record_key order differs from the aligned reference")
if not np.array_equal(aligned_manifest["call_id"].to_numpy(), np.asarray(calls)):
    raise ValueError("STOP: preflight call_id order differs from the aligned reference")
if not np.array_equal(aligned_manifest["speaker"].to_numpy(), np.asarray(labels)):
    raise ValueError("STOP: preflight labels differ from the aligned reference")

sample_lengths = (
    aligned_manifest["crop_end_frame"].to_numpy(dtype=np.int64)
    - aligned_manifest["crop_start_frame"].to_numpy(dtype=np.int64)
)
if np.any(sample_lengths <= 0):
    raise ValueError("STOP: manifest contains non-positive crop lengths")

print("Training rows:", len(sample_lengths))
print("Minimum sample length:", int(sample_lengths.min()))
print("Sample length quantiles (samples):")
display(pd.Series(sample_lengths).quantile([0, 0.001, 0.01, 0.05, 0.5, 0.9, 1]))
length_thresholds = (80, 160, 320, 400, 800, 1600, 3200)
display(pd.DataFrame({
    "at_most_samples": length_thresholds,
    "row_count": [int(np.count_nonzero(sample_lengths <= target)) for target in length_thresholds],
}))

# 길이와 record_key로 정렬해 같은 manifest에서 동일한 실제 발화가 선택되게 한다.
shortest_rows = (
    aligned_manifest.assign(sample_length=sample_lengths)
    .sort_values(["sample_length", "record_key"], kind="stable")
    .head(100)
)
preflight_records = []
failed_waveforms = []
for row in shortest_rows.itertuples(index=False):
    audio = read_wav_frames(RAW_DATA_ROOT / str(row.audio_path))
    if audio.sample_rate != 8_000 or audio.waveform.shape[0] != 1:
        raise ValueError(f"STOP: unsupported WAV for {row.record_key}")
    start, end = int(row.crop_start_frame), int(row.crop_end_frame)
    if start < 0 or end > audio.waveform.shape[1] or end <= start:
        raise ValueError(f"STOP: invalid crop for {row.record_key}")
    crop = audio.waveform[0, start:end]
    if len(crop) != int(row.sample_length):
        raise ValueError(f"STOP: crop length mismatch for {row.record_key}")

    # NaN을 보정하지 않고 실패로 기록한다. 경고만 숨겨 표에 행별 결과를 남긴다.
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Segment too short")
        try:
            vector, names = extract_egemaps(crop, audio.sample_rate)
            success = (
                vector.shape == (88,)
                and np.isfinite(vector).all()
                and names == feature_names_88
            )
            failure_reason = None if success else "88D / finite / feature-name contract mismatch"
        except ValueError as error:
            success = False
            failure_reason = str(error)
    if not success and len(failed_waveforms) < 5:
        failed_waveforms.append((str(row.record_key), crop.copy()))
    preflight_records.append({
        "record_key": str(row.record_key),
        "call_id": str(row.call_id),
        "label": int(row.speaker),
        "sample_length": len(crop),
        "duration_seconds": len(crop) / audio.sample_rate,
        "success": bool(success),
        "failure_reason": failure_reason,
    })

preflight_rows = pd.DataFrame(preflight_records)
print("Real short utterances tested:", len(preflight_rows))
display(preflight_rows)
failed_rows = preflight_rows.loc[~preflight_rows["success"]]
successful_rows = preflight_rows.loc[preflight_rows["success"]]
print("Failure count among tested rows:", len(failed_rows))
print("Failure sample length min/max:",
      None if failed_rows.empty else (int(failed_rows["sample_length"].min()),
                                     int(failed_rows["sample_length"].max())))
print("Shortest successful sample length:",
      None if successful_rows.empty else int(successful_rows["sample_length"].min()))
print("Longest failed sample length:",
      None if failed_rows.empty else int(failed_rows["sample_length"].max()))

# 실제 실패 발화 최대 5개만 사용한다. 뒤에 0을 붙인 복사본을 시험하며 원본은 보존한다.
padding_targets = (160, 320, 400, 800, 1600, 3200)
padding_records = []
for record_key, crop in failed_waveforms:
    for target in padding_targets:
        padded = np.pad(crop, (0, max(0, target - len(crop))), mode="constant")
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="Segment too short")
            try:
                vector, names = extract_egemaps(padded, 8_000)
                finite = bool(vector.shape == (88,) and np.isfinite(vector).all())
                success_88d = bool(finite and names == feature_names_88)
            except ValueError:
                finite = False
                success_88d = False
        padding_records.append({
            "record_key": record_key,
            "target_samples": target,
            "tested_samples": len(padded),
            "success_88d": success_88d,
            "finite_88d": finite,
        })

padding_probe = pd.DataFrame(padding_records)
print("Actual failed utterances used for padding probe:", len(failed_waveforms))
if padding_probe.empty:
    print("No failed utterance in the shortest 100; no padding candidate was tested.")
else:
    display(padding_probe)
    safe_targets = [
        target for target in padding_targets
        if padding_probe.loc[padding_probe["target_samples"] == target, "success_88d"].all()
    ]
    print("Smallest passing target for all probed failures (not a production policy):",
          safe_targets[0] if safe_targets else None)


Training rows: 889358
Minimum sample length: 72
Sample length quantiles (samples):


0.000        72.0
0.001      1600.0
0.010      2336.0
0.050      3440.0
0.500     11520.0
0.900     33464.0
1.000    193376.0
dtype: float64

,at_most_samples,row_count
0,80,3
1,160,3
2,320,3
3,400,3
4,800,11
5,1600,1062
6,3200,35521


Real short utterances tested: 100


,record_key,call_id,label,sample_length,duration_seconds,success,failure_reason
0,Training|651e4e712e98ee7120968df5_20220704|28,651e4e712e98ee7120968df5_20220704,1,72,0.009,False,eGeMAPSv02 returned NaN or inf; audio may be t...
1,Training|651e4f531e44dfea6eb05a69_20220901|3,651e4f531e44dfea6eb05a69_20220901,0,80,0.010,False,eGeMAPSv02 returned NaN or inf; audio may be t...
2,Training|651e5500835c8a556d7e2a8d_20220404|40,651e5500835c8a556d7e2a8d_20220404,0,80,0.010,False,eGeMAPSv02 returned NaN or inf; audio may be t...
3,Training|651e4f9dddc447a00994574f_20220904|27,651e4f9dddc447a00994574f_20220904,1,560,0.070,True,NaN
4,Training|651e54e7af7ab8d7995ab3a9_20220403|33,651e54e7af7ab8d7995ab3a9_20220403,1,576,0.072,True,NaN
...,...,...,...,...,...,...,...
95,Training|651e4c5b48bae90750379aab_20220602|14,651e4c5b48bae90750379aab_20220602,0,1200,0.150,True,NaN
96,Training|651e4d9be84a30cdf982d59e_20220702|32,651e4d9be84a30cdf982d59e_20220702,1,1200,0.150,True,NaN
97,Training|651e4e712e98ee7120968fa2_20220704|38,651e4e712e98ee7120968fa2_20220704,0,1200,0.150,True,NaN
98,Training|651e4e89429d02dab45a2af9_20220707|18,651e4e89429d02dab45a2af9_20220707,1,1200,0.150,True,NaN


Failure count among tested rows: 3
Failure sample length min/max: (72, 80)
Shortest successful sample length: 560
Longest failed sample length: 80
Actual failed utterances used for padding probe: 3


,record_key,target_samples,tested_samples,success_88d,finite_88d
0,Training|651e4e712e98ee7120968df5_20220704|28,160,160,False,False
1,Training|651e4e712e98ee7120968df5_20220704|28,320,320,False,False
2,Training|651e4e712e98ee7120968df5_20220704|28,400,400,False,False
3,Training|651e4e712e98ee7120968df5_20220704|28,800,800,True,True
4,Training|651e4e712e98ee7120968df5_20220704|28,1600,1600,True,True
5,Training|651e4e712e98ee7120968df5_20220704|28,3200,3200,True,True
6,Training|651e4f531e44dfea6eb05a69_20220901|3,160,160,False,False
7,Training|651e4f531e44dfea6eb05a69_20220901|3,320,320,False,False
8,Training|651e4f531e44dfea6eb05a69_20220901|3,400,400,False,False
9,Training|651e4f531e44dfea6eb05a69_20220901|3,800,800,True,True


Smallest passing target for all probed failures (not a production policy): 800


## Checkpoint D — validated partial에서 전체 Training 88D 재개·최종 cache 검증

이 코드 셀을 직접 실행할 때만 남은 행을 추출한다. 최종 cache가 없으면 기존 partial recovery의 행 순서·완료 mask·finite 값·feature 순서·clean provenance를 먼저 검증하고 완료 행과 완료된 call의 WAV 읽기를 건너뛴다. 원본 crop으로 먼저 추출하며, 실제 short-segment nonfinite 실패에만 현재 발화 뒤를 zero-pad-to-800으로 채워 한 번 재시도한다. 250 calls마다 mmap과 metadata를 저장한다. 전체 행이 검증된 뒤에만 최종 cache를 만든다. **아래에 보이는 이전 D 출력은 이전 실패 실행의 기록이며, 수정된 코드의 실행 결과가 아니다.**


In [24]:
import importlib
import sys

import dcc_data.mission2_standard_representation as msr
msr = importlib.reload(msr)

# 실패했던 resume module도 최신 상태로 다시 import
sys.modules.pop("dcc_data.mission2_egemaps_resume", None)
import dcc_data.mission2_egemaps_resume as mer

# Notebook에서 D가 쓰는 함수도 최신 버전으로 교체
load_egemaps_cache = msr.load_egemaps_cache

print("_padded_rows_from_metadata exists:",
      hasattr(msr, "_padded_rows_from_metadata"))
print("resume module loaded:", mer.SHORT_AUDIO_FALLBACK)

_padded_rows_from_metadata exists: True
resume module loaded: zero_pad_to_800_on_nonfinite_short_segment


In [25]:
from dcc_data.mission2_egemaps_resume import (
    SHORT_AUDIO_FALLBACK, resume_egemaps_extraction,
)

RECOVERY_ROOT = REPO_ROOT / "tmp" / "mission2_egemaps_partial_recovery"
if source_provenance != CLEAN_REFERENCE_FINGERPRINT:
    raise ValueError("STOP: current clean fingerprint differs from the frozen reference")
if EGEMAPS_CACHE_PATH.is_file():
    egemaps_data = load_egemaps_cache(
        EGEMAPS_CACHE_PATH, keys, calls, source_provenance=source_provenance,
    )
else:
    egemaps_data = resume_egemaps_extraction(
        aligned_manifest, RAW_DATA_ROOT, RECOVERY_ROOT, EGEMAPS_CACHE_PATH,
        keys, calls, feature_names_88,
        source_provenance=source_provenance,
        clean_reference_fingerprint=CLEAN_REFERENCE_FINGERPRINT,
    )
if (egemaps_data.metadata.get("short_audio_fallback") != SHORT_AUDIO_FALLBACK
        or egemaps_data.metadata.get("padding_target_samples") != 800
        or egemaps_data.metadata.get("sample_rate") != 8_000):
    raise ValueError("STOP: final cache does not record the approved short-audio policy")
print("Validated cache:", EGEMAPS_CACHE_PATH)
print("Shape / dtype:", egemaps_data.X.shape, egemaps_data.X.dtype)
print("Padded rows:", egemaps_data.metadata["padded_row_count"])
print("Cleanup candidate after final validation (not deleted):", RECOVERY_ROOT)
display(pd.DataFrame([dict(egemaps_data.metadata)]))


REUSING_PARTIAL_EGEMAPS_CACHE
Completed / remaining: 353670 535688
Calls visited: 11750 completed: 357050 remaining: 532308 padded: 1
Calls visited: 12000 completed: 364395 remaining: 524963 padded: 1
Calls visited: 12250 completed: 371431 remaining: 517927 padded: 1
Calls visited: 12500 completed: 378891 remaining: 510467 padded: 1
Calls visited: 12750 completed: 386558 remaining: 502800 padded: 1
Calls visited: 13000 completed: 393835 remaining: 495523 padded: 1
Calls visited: 13250 completed: 401365 remaining: 487993 padded: 1
Calls visited: 13500 completed: 408993 remaining: 480365 padded: 1
Calls visited: 13750 completed: 417111 remaining: 472247 padded: 1
Calls visited: 14000 completed: 425079 remaining: 464279 padded: 1
Calls visited: 14250 completed: 432687 remaining: 456671 padded: 1
Calls visited: 14500 completed: 440491 remaining: 448867 padded: 2
Calls visited: 14750 completed: 447739 remaining: 441619 padded: 2
Calls visited: 15000 completed: 455377 remaining: 433981 padde

KeyboardInterrupt: 

In [26]:
import numpy as np

saved_matrix = np.load(
    REPO_ROOT / "tmp" / "mission2_egemaps_partial_recovery" / "matrix_88_partial.npy",
    mmap_mode="r",
)
saved_completed = np.isfinite(saved_matrix).all(axis=1)

print("saved completed:", int(saved_completed.sum()))
print("saved remaining:", int((~saved_completed).sum()))

saved completed: 470255
saved remaining: 419103


In [17]:
completed_mask = np.isfinite(matrix_88).all(axis=1)

print("matrix shape:", matrix_88.shape)
print("completed rows:", int(completed_mask.sum()))
print("remaining rows:", int((~completed_mask).sum()))
print("completion rate:", float(completed_mask.mean()))

matrix shape: (889358, 88)
completed rows: 353670
remaining rows: 535688
completion rate: 0.3976688802484489


In [18]:
from pathlib import Path
import json
import numpy as np

PARTIAL_DIR = REPO_ROOT / "tmp" / "mission2_egemaps_partial_recovery"
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)

np.save(PARTIAL_DIR / "matrix_88_partial.npy", matrix_88)
np.save(PARTIAL_DIR / "completed_mask.npy", completed_mask)

metadata = {
    "rows": int(matrix_88.shape[0]),
    "dim": int(matrix_88.shape[1]),
    "completed_rows": int(completed_mask.sum()),
    "remaining_rows": int((~completed_mask).sum()),
    "feature_names": list(feature_names_88),
    "clean_reference_fingerprint":
        "49d65a35d1e98c280f98e45cbed864ef8bac36268ccffbe81aa0985923c90de3",
}

with open(PARTIAL_DIR / "recovery_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("RECOVERY SAVED:", PARTIAL_DIR)

RECOVERY SAVED: C:\Users\YW\Study\02_Projects\DCC_2026\prelim\tmp\mission2_egemaps_partial_recovery


In [19]:
np.save(PARTIAL_DIR / "record_keys.npy", np.asarray(keys))
np.save(PARTIAL_DIR / "call_ids.npy", np.asarray(calls))

print("alignment metadata saved")

alignment metadata saved


## Checkpoint E — 두 representation의 공정 비교 guard

두 입력의 행·label·call·split과 M04-100 설정을 fit 전에 확인한다. 59D는 지정된 clean chain fingerprint, B4 Accuracy `0.873760` 및 CM까지 확인한다. scaler는 representation마다 같은 internal Training 행에서만 fit한다. 평가는 두 결과 모두 기존 calculate_prediction_metrics를 사용한다.


In [ ]:
expected_m04 = {**dict(M04_MLP_CONFIG), "max_iter": 100}
checks = {
    "clean_chain_fingerprint": (clean_59d.provenance_fingerprint == source_provenance
                                == CLEAN_REFERENCE_FINGERPRINT),
    "clean_chain_path": clean_59d.chain_root == CLEAN_CHAIN_ROOT,
    "clean_59d_matrix": (clean_59d.X.shape == (len(keys), 59)
                          and clean_59d.X.dtype == np.float32
                          and np.isfinite(clean_59d.X).all()),
    "clean_59d_rows_labels": (np.array_equal(clean_59d.record_keys, keys)
                               and np.array_equal(clean_59d.call_ids, calls)
                               and np.array_equal(clean_59d.y, labels)),
    "clean_59d_feature_order": prepared_59d.feature_names == clean_59d.feature_names,
    "clean_59d_accuracy_cm": (
        abs(result_59d.metrics["accuracy"] - CLEAN_REFERENCE_ACCURACY) <= 1e-12
        and np.array_equal(np.asarray(result_59d.metrics["confusion_matrix"]),
                           np.asarray(CLEAN_REFERENCE_CM))
        and result_59d.metrics["correct_count"] == 155434
        and report_59d["provenance_fingerprint"] == CLEAN_REFERENCE_FINGERPRINT
    ),
    "clean_59d_fit_audit": (
        np.array_equal(result_59d.fit_audit["train_idx"], train_idx)
        and np.array_equal(result_59d.fit_audit["validation_idx"], validation_idx)
        and np.array_equal(result_59d.y_true, labels[validation_idx])
        and result_59d.fit_audit["scaler_fit_scope"] == "internal_train_only"
        and result_59d.fit_audit["seed"] == 42
        and dict(result_59d.model_config) == expected_m04
    ),
    "88d_source_provenance": (
        egemaps_data.metadata["source_provenance"] == CLEAN_REFERENCE_FINGERPRINT
    ),
    "record_key": np.array_equal(egemaps_data.record_keys, keys),
    "labels_59d_train": np.array_equal(prepared_59d.y_train, labels[train_idx]),
    "labels_59d_validation": np.array_equal(
        prepared_59d.y_evaluation, labels[validation_idx]
    ),
    "call_id": np.array_equal(egemaps_data.call_ids, calls),
    "train_indices": np.array_equal(prepared_59d.train_idx, train_idx),
    "validation_indices": np.array_equal(
        prepared_59d.validation_idx, validation_idx
    ),
    "seed": split_audit["seed"] == expected_m04["random_state"] == 42,
    "model_config": dict(M04_ITER100_FEATURE_SCREENING_CONFIG) == expected_m04,
    "59d_scaler_fit_rows": int(prepared_59d.scaler.n_samples_seen_) == len(train_idx),
    "88d_feature_order": egemaps_data.feature_names == feature_names_88,
    "88d_shape_dtype_finite": (
        egemaps_data.X.shape == (len(keys), 88)
        and egemaps_data.X.dtype == np.float32
        and np.isfinite(egemaps_data.X).all()
    ),
    "split_audit": dict(prepared_59d.split_audit) == split_audit,
}
display(pd.DataFrame([
    {"contract": name, "pass": bool(passed)}
    for name, passed in checks.items()
]))
if not all(checks.values()):
    raise RuntimeError("STOP: fair comparison input contract differs")


## Checkpoint F — clean 59D M04-100 internal reference (`0.873760`)

E의 strict 비교 guard를 통과한 B4 clean 59D 결과만 다시 표시한다. 모델을 중복 fit하지 않는다. historical `0.874558`은 여기서 사용하지 않는다.


In [ ]:
if "result_59d" not in globals():
    raise RuntimeError("STOP: run Checkpoint B4 first")
if "checks" not in globals():
    raise RuntimeError("STOP: run Checkpoint E before F")
if not all(checks.values()):
    raise RuntimeError("STOP: run Checkpoint E with the clean 59D reference")
display(pd.DataFrame([report_59d]))

## Checkpoint G — eGeMAPSv02 88D, 동일 M04-100

기존 handcrafted 준비 helper는 새 feature 이름을 의도적으로 거부한다. 여기서는 같은 StandardScaler를 같은 internal Training index에 fit해 기존 PreparedRepresentation에 담고, F와 동일한 runner를 호출한다. 이 셀을 직접 실행할 때만 88D 모델을 fit한다.


In [ ]:
scaler_88 = StandardScaler()
X_train_88 = scaler_88.fit_transform(
    egemaps_data.X[train_idx]
).astype(np.float32, copy=False)
X_validation_88 = scaler_88.transform(
    egemaps_data.X[validation_idx]
).astype(np.float32, copy=False)
if int(scaler_88.n_samples_seen_) != len(train_idx):
    raise RuntimeError("STOP: 88D scaler was not fit on internal Training only")
if not np.isfinite(X_train_88).all() or not np.isfinite(X_validation_88).all():
    raise ValueError("STOP: scaled 88D matrix contains NaN or inf")
prepared_88d = PreparedRepresentation(
    representation_id=REPRESENTATION_ID,
    feature_names=egemaps_data.feature_names,
    X_train_scaled=X_train_88,
    X_evaluation_scaled=X_validation_88,
    y_train=labels[train_idx],
    y_evaluation=labels[validation_idx],
    training_record_keys=keys[train_idx],
    evaluation_record_keys=keys[validation_idx],
    train_idx=train_idx.copy(),
    validation_idx=validation_idx.copy(),
    scaler=scaler_88,
    split_audit=split_audit,
)
result_88d = _run_prepared_representation(
    prepared_88d, experiment_id=REPRESENTATION_ID, allow_fit=True
)
if dict(result_88d.model_config) != expected_m04:
    raise RuntimeError("STOP: eGeMAPS M04-100 configuration changed")
if result_88d.fit_audit["scaler_fit_scope"] != "internal_train_only":
    raise RuntimeError("STOP: 88D scaler fit scope changed")
train_metrics_88d = calculate_prediction_metrics(
    prepared_88d.y_train,
    result_88d.model.predict(prepared_88d.X_train_scaled),
)
report_88d = {
    "representation": REPRESENTATION_ID,
    "dimension": 88,
    "train_accuracy": train_metrics_88d["accuracy"],
    "internal_validation_accuracy": result_88d.metrics["accuracy"],
    "generalization_gap": (
        train_metrics_88d["accuracy"] - result_88d.metrics["accuracy"]
    ),
}
display(pd.DataFrame([report_88d]))


## Checkpoint H — paired fair comparison

clean 59D `0.873760` 기준과 fit audit를 다시 확인한 후 같은 validation 행의 prediction을 짝지어 비교한다. historical `0.874558`과 Delta를 계산하지 않는다. Accuracy 차이와 오류 건수는 관찰값이며 원인이나 우열 판단은 여기서 하지 않는다.


In [ ]:
if (clean_59d.provenance_fingerprint != CLEAN_REFERENCE_FINGERPRINT
        or source_provenance != CLEAN_REFERENCE_FINGERPRINT
        or not np.array_equal(np.asarray(result_59d.metrics["confusion_matrix"]),
                              np.asarray(CLEAN_REFERENCE_CM))
        or abs(result_59d.metrics["accuracy"] - CLEAN_REFERENCE_ACCURACY) > 1e-12):
    raise RuntimeError("STOP: paired comparison requires the clean 59D reference")
for label, result in (("59D", result_59d), ("eGeMAPS", result_88d)):
    audit = result.fit_audit
    if (
        not np.array_equal(audit["train_idx"], train_idx)
        or not np.array_equal(audit["validation_idx"], validation_idx)
        or not np.array_equal(audit["training_record_keys"], keys[train_idx])
        or not np.array_equal(audit["validation_record_keys"], keys[validation_idx])
        or not np.array_equal(result.y_true, labels[validation_idx])
        or audit["seed"] != 42
        or audit["scaler_fit_scope"] != "internal_train_only"
        or dict(result.model_config) != expected_m04
    ):
        raise RuntimeError(f"STOP: {label} fit/audit comparison contract differs")
if not np.array_equal(result_59d.y_true, result_88d.y_true):
    raise RuntimeError("STOP: validation labels differ")
paired = calculate_paired_prediction_changes(
    y_true=labels[validation_idx],
    m00_pred=result_59d.y_pred,
    m01_pred=result_88d.y_pred,
    m00_record_keys=result_59d.fit_audit["validation_record_keys"],
    m01_record_keys=result_88d.fit_audit["validation_record_keys"],
)
paired_summary = {
    "both_correct": paired["both_correct"],
    "both_wrong": paired["both_wrong"],
    "59D_wrong_to_eGeMAPS_correct": paired["m00_wrong_to_m01_correct"],
    "59D_correct_to_eGeMAPS_wrong": paired["m00_correct_to_m01_wrong"],
    "net_correct_gain": (
        paired["m00_wrong_to_m01_correct"]
        - paired["m00_correct_to_m01_wrong"]
    ),
}
comparison_metrics = pd.DataFrame([
    {
        "representation": label,
        "accuracy": result.metrics["accuracy"],
        "confusion_matrix": result.metrics["confusion_matrix"],
        "recall_speaker_0": result.metrics["recall_speaker_0"],
        "recall_speaker_1": result.metrics["recall_speaker_1"],
    }
    for label, result in (("59D", result_59d), ("eGeMAPS", result_88d))
])
display(comparison_metrics)
display(pd.DataFrame([paired_summary]))
print("Internal Validation Accuracy delta vs clean 59D:",
      result_88d.metrics["accuracy"] - result_59d.metrics["accuracy"])


## Checkpoint I — duration·overlap·confidence 진단

기존 진단 helper를 clean 59D와 eGeMAPS 결과에 똑같이 적용한다. duration, absolute crop 위치와 overlap은 이 셀의 진단용 tracking 정보이며 모델 feature가 아니다. 각 모델의 confidence는 calibration이 다를 수 있으므로 직접 비교할 때 주의한다.


In [ ]:
training_tracking = tracking.iloc[train_idx].reset_index(drop=True)
validation_tracking = tracking.iloc[validation_idx].reset_index(drop=True)
diagnostics = {
    name: build_standalone_diagnostic_report(
        result, training_tracking=training_tracking,
        evaluation_tracking=validation_tracking,
    )
    for name, result in (("59D", result_59d), ("eGeMAPS", result_88d))
}
for slice_name in ("duration_summary", "overlap_summary", "confidence_summary"):
    print(slice_name)
    display(pd.concat([
        diagnostics[name][slice_name].assign(representation=name)
        for name in ("59D", "eGeMAPS")
    ], ignore_index=True))
display(pd.DataFrame([
    {"representation": name,
     "high_confidence_error_count": diagnostics[name]["high_confidence_error_count"]}
    for name in ("59D", "eGeMAPS")
]))

## Checkpoint J — 결과 표와 사용자 해석

clean 59D `0.873760` 기준과 eGeMAPS의 관찰값만 출력한다. historical `0.874558`은 비교 표에 포함하지 않는다. 아래 빈 항목은 사용자가 실행 결과를 보고 직접 작성한다.


In [ ]:
summary_table = pd.DataFrame([report_59d, report_88d])
display(summary_table)
display(comparison_metrics)
display(pd.DataFrame([paired_summary]))
print("Cache:", EGEMAPS_CACHE_PATH)
print("Cache bytes:", EGEMAPS_CACHE_PATH.stat().st_size)
print("Result interpretation: write only after reviewing the observed outputs.")


### 사용자 해석 메모

- 관찰:
- clean 59D(Internal Accuracy 0.873760) 대비 eGeMAPS 변화:
- class별 변화:
- 주요 error 변화:
- generalization gap 변화:
- 비용 변화:
- leakage / comparison integrity 확인:
- Stage 5로 가져갈 가설:
- 다음 판단:
